# DATA PREPROCESSING AND FEATURE ENGINEERING IN MACHINE LEARNING

### 1. Data Exploration and Scaling

##### Data Exploration
The dataset contains 32,561 entries and 15 columns. While .isnull().sum() initially returns 0, missing values in this dataset are represented by the string ' ?'.

Handling Missing Values: I replaced ' ?' with NaN and imputed categorical columns with the mode and numerical columns with the mean.

##### Scaling Techniques
1.Standard Scaling: Rescales data to have a mean of 0 and a standard deviation of 1.

2.Min-Max Scaling: Scales data to a fixed range, usually [0, 1].

### 2. Encoding Techniques
##### Implementation
One-Hot Encoding: Applied to sex and income as they have fewer than 5 categories.

Label Encoding: Applied to variables like workclass, education, and native_country which have more than 5 categories to avoid creating a massive number of sparse columns.

##### Pros and Cons:

One-Hot Encoding:

Pros: Does not assume an arbitrary order between categories; highly effective for linear models.

Cons: Can lead to the "Curse of Dimensionality" if a feature has many unique categories.

##### Label Encoding:

Pros: Memory efficient; keeps the number of features constant.

Cons: May lead the model to believe there is a mathematical hierarchy between categories (e.g., treating 'Private' as greater than 'Self-emp') where none exists.

### 3. Feature Engineering
##### New Features Created
Capital Diff (capital_diff): Calculated as capital_gain - capital_loss.

Rationale: This provides a net view of an individual's financial growth or loss in a single feature, which is likely a strong predictor of income level.

Age Group (age_group): Categorized age into four bins: Young, Middle-aged, Senior, and Retired.

Rationale: Income often follows a life-cycle pattern where it peaks in middle age and decreases during retirement; binning helps capture these non-linear relationships.

##### Skewness Transformation
Applied Log Transformation (np.log1p) to capital_gain.

Rationale: capital_gain is highly right-skewed with many zeros and a few very large values. Log transformation compresses the range of large values and makes the distribution more "normal," which helps many machine learning algorithms converge faster.

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

# 1. Load the dataset
df = pd.read_csv('adult_with_headers.csv')

# 2. Data Cleaning: Handle ' ?' missing values
df.replace(' ?', np.nan, inplace=True)

# Impute: Mode for categorical, Mean for numerical
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].mean())

# 3. Scaling (Preparation)
num_cols = ['age', 'education_num', 'capital_gain', 'hours_per_week']
scaler_std = StandardScaler()
scaler_mm = MinMaxScaler()

# Standardized and Min-Max Scaled versions
df_std = pd.DataFrame(scaler_std.fit_transform(df[num_cols]), columns=[c+'_std' for c in num_cols])
df_minmax = pd.DataFrame(scaler_mm.fit_transform(df[num_cols]), columns=[c+'_minmax' for c in num_cols])

# 4. Feature Engineering
# Create capital difference
df['capital_diff'] = df['capital_gain'] - df['capital_loss']

# Create log transformation for capital_gain
df['log_capital_gain'] = np.log1p(df['capital_gain'])

# Create age groups (binning)
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 45, 65, 100], labels=['Young', 'Middle-aged', 'Senior', 'Retired'])

# 5. Encoding
# Final dataframe copy
df_final = df.copy()

# Identify categorical and newly created binned columns
cat_cols = df_final.select_dtypes(include=['object', 'category']).columns

for col in cat_cols:
    if df_final[col].nunique() < 5:
        # One-Hot Encoding for small categories (like sex, income, age_group)
        df_final = pd.get_dummies(df_final, columns=[col], drop_first=True)
    else:
        # Label Encoding for larger categories (like workclass, occupation)
        le = LabelEncoder()
        df_final[col] = le.fit_transform(df_final[col].astype(str))

# Display Output
print("--- Processed Data (First 5 Rows) ---")
print(df_final.head())

# Save result
df_final.to_csv('Processed_Adult_Data.csv', index=False)

--- Processed Data (First 5 Rows) ---
   age  workclass  fnlwgt  education  education_num  marital_status  \
0   39          6   77516          9             13               4   
1   50          5   83311          9             13               2   
2   38          3  215646         11              9               0   
3   53          3  234721          1              7               2   
4   28          3  338409          9             13               2   

   occupation  relationship  race  capital_gain  capital_loss  hours_per_week  \
0           0             1     4          2174             0              40   
1           3             0     4             0             0              13   
2           5             1     4             0             0              40   
3           5             0     2             0             0              40   
4           9             5     2             0             0              40   

   native_country  capital_diff  log_capital_gai